<a href="https://colab.research.google.com/github/TrinaBan0807/python-starter-kit/blob/main/Clip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install open_clip_torch
import torch
import open_clip
import copy
import numpy as np

# --- 1. Configuration: Multiple Models and Label Sets ---
# You can add or remove models and label versions here
model_configs = [
    ('ViT-B-32', 'openai'),
    ('ViT-L-14', 'openai')  # Evaluating larger scale to check for bias amplification
]

# Different label sets to test "Class Design" impact
label_scenarios = {
    "Broad_Labels": ["cat", "dog", "car", "truck"],
    "Refined_Labels": ["Siamese cat", "Golden Retriever", "sports car", "dump truck"],
    "Ethical_Taxonomy": ["child", "person", "human", "individual"] # Testing 'safe' class patterns
}

# Range of alpha values to address the Sample Efficiency Gap
test_alphas = [0.0, 0.3, 0.7, 1.0]

# --- 2. Evaluation Engine ---
for model_name, pretrain in model_configs:
    print(f"\n{'='*60}\nINITIALIZING MODEL: {model_name}\n{'='*60}")

    # Load environment
    model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained=pretrain)
    tokenizer = open_clip.get_tokenizer(model_name)
    model.eval()

    # Weight Interpolation Preparation
    theta_0 = model.text_projection.clone().detach()
    theta_few = theta_0 + 0.02 * torch.randn_like(theta_0) # Simulated few-shot update

    for scenario_name, labels in label_scenarios.items():
        print(f"\n--- Scenario: {scenario_name} ---")
        print(f"{'Alpha':<8} | {'Top Prediction':<20} | {'Confidence':<10}")
        print("-" * 45)

        # Prepare text and dummy image
        text_tokens = tokenizer(labels)
        dummy_image = torch.randn(1, model.visual.output_dim)
        dummy_image /= dummy_image.norm(dim=-1, keepdim=True)

        for alpha in test_alphas:
            with torch.no_grad():
                # Mathematical tweak: (1 - alpha) * theta_0 + alpha * theta_few
                interpolated_weight = (1 - alpha) * theta_0 + alpha * theta_few

                # Update model projection
                model.text_projection.copy_(interpolated_weight)

                # Calculate Result
                text_features = model.encode_text(text_tokens)
                text_features /= text_features.norm(dim=-1, keepdim=True)

                logits = (dummy_image @ text_features.T).softmax(dim=-1)
                prob, idx = torch.max(logits[0], dim=0)

                print(f"{alpha:<8.1f} | {labels[idx.item()]:<20} | {prob.item():.4f}")


INITIALIZING MODEL: ViT-B-32

--- Scenario: Broad_Labels ---
Alpha    | Top Prediction       | Confidence
---------------------------------------------
0.0      | car                  | 0.2542
0.3      | dog                  | 0.2541
0.7      | dog                  | 0.2580
1.0      | dog                  | 0.2597

--- Scenario: Refined_Labels ---
Alpha    | Top Prediction       | Confidence
---------------------------------------------
0.0      | Golden Retriever     | 0.2673
0.3      | Golden Retriever     | 0.2647
0.7      | Golden Retriever     | 0.2603
1.0      | Golden Retriever     | 0.2576

--- Scenario: Ethical_Taxonomy ---
Alpha    | Top Prediction       | Confidence
---------------------------------------------
0.0      | person               | 0.2539
0.3      | person               | 0.2535
0.7      | person               | 0.2527
1.0      | person               | 0.2522

INITIALIZING MODEL: ViT-L-14


open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]


--- Scenario: Broad_Labels ---
Alpha    | Top Prediction       | Confidence
---------------------------------------------
0.0      | cat                  | 0.2541
0.3      | cat                  | 0.2537
0.7      | cat                  | 0.2531
1.0      | cat                  | 0.2526

--- Scenario: Refined_Labels ---
Alpha    | Top Prediction       | Confidence
---------------------------------------------
0.0      | Golden Retriever     | 0.2534
0.3      | Golden Retriever     | 0.2533
0.7      | sports car           | 0.2551
1.0      | sports car           | 0.2573

--- Scenario: Ethical_Taxonomy ---
Alpha    | Top Prediction       | Confidence
---------------------------------------------
0.0      | human                | 0.2520
0.3      | human                | 0.2545
0.7      | human                | 0.2570
1.0      | human                | 0.2583
